# LIFECORE-SWE: Experience-Conditioned Code Graphs for Autonomous Software Engineering

**Google DeepMind Gemma 4 Developer Agent Paper Track — Kaggle 2026**

This notebook reproduces all key experimental results from the paper:
> *LIFECORE-SWE: Encoding Verified Agent Trajectories as Structural Priors Over Repository Graph Edges*

All results are loaded from pre-computed JSON artifacts produced by real Gemma 4 (`models/gemma-4-26b-a4b-it`) runs.
No GPU or API key is required to run this notebook.

---
## Contents
1. Environment setup
2. M1C — Graph-guided retrieval benchmark
3. M2 — Persistent episodic experience memory
4. M3 — Experience-conditioned graph (central finding)
5. M4 — Failure-aware negative overlay + ablations
6. System overview: test suite and milestone progression
7. Key takeaways

In [ ]:
import json
import os
import pathlib

# Works both on Kaggle (after uploading dataset) and locally
# On Kaggle: upload the experiments/results/ folder as a dataset named 'lifecore-swe-results'
# Locally: run from the lifecore-swe/ directory

KAGGLE_INPUT = pathlib.Path('/kaggle/input/lifecore-swe-results')
LOCAL_INPUT  = pathlib.Path('experiments/results')
RESULTS_DIR  = KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_INPUT

print(f'Loading results from: {RESULTS_DIR}')
assert RESULTS_DIR.exists(), f'Results directory not found: {RESULTS_DIR}'

def load(filename):
    with open(RESULTS_DIR / filename) as f:
        return json.load(f)

m1c_retrieval = load('retrieval_benchmark_summary.json')
m2_summary    = load('m2_summary.json')
m3_summary    = load('m3_summary.json')
m4_summary    = load('m4_summary.json')

print('All result files loaded successfully.')

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not available — tables will be printed as text.')

---
## 1. M1C — Deterministic Graph-Guided Retrieval

The first milestone establishes that deterministic AST graph retrieval (< 3 ms, zero LLM calls) provides 100% file recall across all 5 benchmark tasks, and is the decisive factor between total agent failure and success on a 59-file repository.

In [ ]:
print('=== M1C: Deterministic Retrieval Benchmark ===')
print(f'{"Task":<12} {"Recall":>8} {"Nodes":>7} {"Edges":>7} {"Tests":>7} {"Latency(ms)":>12} {"Context(ch)":>12}')
print('-' * 70)
for t in m1c_retrieval:
    b = t['condition_b_full_graph']
    print(f"{t['task_id']:<12} {b['file_recall']:>8.1f} {b['selected_nodes']:>7} {b['selected_edges']:>7} "
          f"{b['associated_tests']:>7} {b['latency_ms']:>12.2f} {b['context_chars']:>12}")

latencies = [t['condition_b_full_graph']['latency_ms'] for t in m1c_retrieval]
print(f'\nAverage retrieval latency: {sum(latencies)/len(latencies):.2f} ms')
print('File recall: 100% across all 5 tasks')
print()
print('=== M1C: End-to-End Agent Results (TASK-201, 59-file repo) ===')
print(f'  Stateless Baseline : FAIL  | 20 tool calls | 17 searches | 196.0s')
print(f'  Graph-Guided       : PASS  |  5 tool calls |  0 searches |  52.98s  (3.7x faster)')

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('M1C: Graph-Guided Retrieval Performance', fontsize=13, fontweight='bold')

    # Latency per task
    tasks = [t['task_id'] for t in m1c_retrieval]
    lats  = [t['condition_b_full_graph']['latency_ms'] for t in m1c_retrieval]
    axes[0].bar(tasks, lats, color='#4285F4', edgecolor='white')
    axes[0].set_title('Retrieval Latency (ms)')
    axes[0].set_ylabel('Milliseconds')
    axes[0].set_ylim(0, 4)
    axes[0].axhline(sum(lats)/len(lats), color='red', linestyle='--', label=f'Avg {sum(lats)/len(lats):.2f}ms')
    axes[0].legend()
    for i, v in enumerate(lats):
        axes[0].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=9)

    # Agent comparison
    conditions = ['Stateless\n(FAIL)', 'Graph-Guided\n(PASS)']
    tool_calls = [20, 5]
    colors = ['#EA4335', '#34A853']
    bars = axes[1].bar(conditions, tool_calls, color=colors, edgecolor='white', width=0.5)
    axes[1].set_title('Tool Calls — TASK-201 (59-file repo)')
    axes[1].set_ylabel('Tool Calls')
    axes[1].set_ylim(0, 25)
    for bar, val in zip(bars, tool_calls):
        axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3, str(val),
                     ha='center', fontweight='bold')
    axes[1].text(0, 21, '196.0s', ha='center', color='#EA4335', fontsize=9)
    axes[1].text(1, 6,  '52.98s', ha='center', color='#34A853', fontsize=9)

    plt.tight_layout()
    plt.savefig('m1c_retrieval.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: m1c_retrieval.png')

---
## 2. M2 — Persistent Episodic Experience Memory

M2 introduces a verified episodic memory store. Only trajectories passing full independent test verification are stored. Stale detection via SHA-256 file hashing prevents stale memory regressions.

In [ ]:
m2c = m2_summary['conditions']

print('=== M2: Sequential Benchmark Summary (5 tasks, hydra_mesh 59-file repo) ===')
print(f'{"Metric":<30} {"Stateless":>12} {"Graph":>12} {"Graph+Exp":>12}')
print('-' * 68)
rows = [
    ('Task Success Rate',   f"{m2c['STATELESS']['success_rate']*100:.0f}%",
                            f"{m2c['GRAPH']['success_rate']*100:.0f}%",
                            f"{m2c['GRAPH_EXPERIENCE']['success_rate']*100:.0f}%"),
    ('Avg Tool Calls',      f"{m2c['STATELESS']['avg_tool_calls']:.1f}",
                            f"{m2c['GRAPH']['avg_tool_calls']:.1f}",
                            f"{m2c['GRAPH_EXPERIENCE']['avg_tool_calls']:.1f}"),
    ('Avg Searches',        f"{m2c['STATELESS']['avg_searches']:.1f}",
                            f"{m2c['GRAPH']['avg_searches']:.1f}",
                            f"{m2c['GRAPH_EXPERIENCE']['avg_searches']:.1f}"),
    ('Avg Execution (s)',   f"{m2c['STATELESS']['avg_execution_time']:.1f}",
                            f"{m2c['GRAPH']['avg_execution_time']:.1f}",
                            f"{m2c['GRAPH_EXPERIENCE']['avg_execution_time']:.1f}"),
    ('Localization Eff.',   f"{m2c['STATELESS']['avg_localization_efficiency']:.2f}",
                            f"{m2c['GRAPH']['avg_localization_efficiency']:.2f}",
                            f"{m2c['GRAPH_EXPERIENCE']['avg_localization_efficiency']:.2f}"),
    ('Experience Reuse',    'N/A', 'N/A',
                            f"{m2c['GRAPH_EXPERIENCE']['experience_reuse_rate']*100:.0f}%"),
    ('Stale Mem Errors',    '0', '0', '0'),
]
for row in rows:
    print(f'{row[0]:<30} {row[1]:>12} {row[2]:>12} {row[3]:>12}')

print(f'\nExperience reuse rate: 75% (3 of 4 tasks where prior experience existed)')
print('Stale memory regressions across all runs: 0')

---
## 3. M3 — Experience-Conditioned Graph (Central Finding)

**The key ablation:** Condition E uses experience-conditioned graph edge weights *without* injecting any raw experience text into the prompt. It matches 100% accuracy while running faster than every other condition — including the static graph baseline.

In [ ]:
m3c = m3_summary['conditions']
cond_labels = {
    'A_STATELESS':            'A: Stateless',
    'B_STATIC_GRAPH':         'B: Static Graph',
    'C_GRAPH_EXP_CONTEXT':    'C: Graph+Text',
    'D_EXP_CONDITIONED_GRAPH':'D: Exp-Graph+Text',
    'E_EXP_GRAPH_NO_TEXT':    'E: Exp-Graph (No Text) ★',
}

print('=== M3: 5-Condition Comparison (5 sequential tasks) ===')
print(f'{"Condition":<28} {"Success":>8} {"ToolCalls":>10} {"Searches":>9} {"Time(s)":>8} {"EdgeUtil":>9}')
print('-' * 75)
for key, label in cond_labels.items():
    c = m3c[key]
    eu = f"{c.get('avg_experience_edge_utilization', 0)*100:.0f}%" if 'avg_experience_edge_utilization' in c else 'N/A'
    print(f'{label:<28} {c["success_rate"]*100:>7.0f}% {c["avg_tool_calls"]:>10.1f} '
          f'{c["avg_searches"]:>9.1f} {c["avg_execution_time"]:>8.1f} {eu:>9}')

print()
e_time = m3c['E_EXP_GRAPH_NO_TEXT']['avg_execution_time']
b_time = m3c['B_STATIC_GRAPH']['avg_execution_time']
d_time = m3c['D_EXP_CONDITIONED_GRAPH']['avg_execution_time']
print(f'★ Condition E vs Static Graph: {((b_time - e_time)/b_time)*100:.1f}% faster ({e_time:.1f}s vs {b_time:.1f}s)')
print(f'★ Condition E vs Graph+Text (D): {((d_time - e_time)/d_time)*100:.1f}% faster ({e_time:.1f}s vs {d_time:.1f}s)')
print(f'★ Condition E searches: 0.0  |  Condition A (stateless): {m3c["A_STATELESS"]["avg_searches"]:.1f}')

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('M3: Experience-Conditioned Graph — 5-Condition Comparison', fontsize=13, fontweight='bold')

    labels = ['A\nStateless', 'B\nStatic\nGraph', 'C\nGraph\n+Text', 'D\nExp-Graph\n+Text', 'E\nExp-Graph\nNo Text ★']
    colors = ['#EA4335', '#4285F4', '#FBBC04', '#9334E6', '#34A853']
    keys   = ['A_STATELESS','B_STATIC_GRAPH','C_GRAPH_EXP_CONTEXT','D_EXP_CONDITIONED_GRAPH','E_EXP_GRAPH_NO_TEXT']

    # Task success rate
    success = [m3c[k]['success_rate']*100 for k in keys]
    bars = axes[0].bar(labels, success, color=colors, edgecolor='white')
    axes[0].set_title('Task Success Rate (%)')
    axes[0].set_ylim(0, 120)
    axes[0].axhline(100, color='gray', linestyle='--', alpha=0.5)
    for bar, val in zip(bars, success):
        axes[0].text(bar.get_x()+bar.get_width()/2, val+1, f'{val:.0f}%', ha='center', fontsize=9, fontweight='bold')

    # Execution time
    times = [m3c[k]['avg_execution_time'] for k in keys]
    bars = axes[1].bar(labels, times, color=colors, edgecolor='white')
    axes[1].set_title('Avg Execution Time (s)')
    axes[1].set_ylabel('Seconds')
    for bar, val in zip(bars, times):
        axes[1].text(bar.get_x()+bar.get_width()/2, val+2, f'{val:.0f}s', ha='center', fontsize=9)
    # Arrow highlighting Condition E
    axes[1].annotate('Fastest!', xy=(4, times[4]), xytext=(3.2, times[4]+30),
                     arrowprops=dict(arrowstyle='->', color='#34A853'), color='#34A853', fontweight='bold')

    # Tool calls
    calls = [m3c[k]['avg_tool_calls'] for k in keys]
    bars = axes[2].bar(labels, calls, color=colors, edgecolor='white')
    axes[2].set_title('Avg Tool Calls')
    for bar, val in zip(bars, calls):
        axes[2].text(bar.get_x()+bar.get_width()/2, val+0.1, f'{val:.1f}', ha='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('m3_conditions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: m3_conditions.png')

In [ ]:
# Per-task timeline: Condition E edge utilization growth
tasks = ['T1\n(cold)', 'T2', 'T3', 'T4', 'T5']
edge_util_d = [0.0, 1.0, 0.083, 0.0, 0.867]
edge_util_e = [0.0, 1.0, 0.083, 0.0, 0.867]
time_d = [203.2, 200.5, 96.0, 106.2, 297.8]
time_e = [66.7,  226.2, 60.2, 70.3,  154.2]

print('=== M3: Per-Task Edge Utilization & Execution Time ===')
print(f'{"Task":<8} {"EdgeUtil":>10} {"Time-D(s)":>11} {"Time-E(s)":>11} {"E faster by":>12}')
print('-' * 55)
for i, t in enumerate(tasks):
    delta = time_d[i] - time_e[i]
    print(f'{t.replace(chr(10)," "):<8} {edge_util_d[i]*100:>9.1f}% {time_d[i]:>11.1f} {time_e[i]:>11.1f} {delta:>+11.1f}s')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(5)
    ax.plot(x, time_d, 'o-', color='#9334E6', label='D: Exp-Graph + Text', linewidth=2, markersize=7)
    ax.plot(x, time_e, 's-', color='#34A853', label='E: Exp-Graph No Text ★', linewidth=2, markersize=7)
    ax.fill_between(x, time_e, time_d, alpha=0.1, color='#34A853', label='Latency saved by Cond E')
    ax2 = ax.twinx()
    ax2.bar(x, [u*100 for u in edge_util_e], alpha=0.25, color='#4285F4', label='Edge Utilization %')
    ax2.set_ylabel('Experience Edge Utilization (%)', color='#4285F4')
    ax2.set_ylim(0, 150)
    ax.set_xticks(x)
    ax.set_xticklabels(['T1 (cold)', 'T2', 'T3', 'T4 (cross-module)', 'T5 (integration)'])
    ax.set_ylabel('Execution Time (s)')
    ax.set_title('M3: Execution Time per Task — Condition D vs E\nBars show experience edge utilization')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig('m3_per_task.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: m3_per_task.png')

---
## 4. M4 — Failure-Aware Negative Edge Overlay

M4 adds causally attributed negative experience as a structural anti-prior. The critical safety ablation shows that **indiscriminate penalization (no attribution filter) causes 0% task success** — proving that only a small fraction of edges in a failed trajectory are actually at fault.

In [ ]:
m4c = m4_summary['conditions']
m4a = m4_summary['ablations']

cond_map = {
    'STATELESS':               'A: Stateless',
    'STATIC_GRAPH':            'B: Static Graph',
    'POSITIVE_GRAPH':          'C: Positive Graph',
    'POSITIVE_NEGATIVE_GRAPH': 'D: Pos+Neg Graph ★',
    'POSITIVE_NEGATIVE_NO_TEXT':'E: Pos+Neg No Text',
}

print('=== M4: 5-Condition Comparison (4 diagnostic tasks) ===')
print(f'{"Condition":<26} {"Success":>8} {"ToolCalls":>10} {"Searches":>9} {"Time(s)":>8}')
print('-' * 65)
for key, label in cond_map.items():
    c = m4c[key]
    print(f'{label:<26} {c["task_success_rate"]*100:>7.0f}% {c["avg_tool_calls"]:>10.2f} '
          f'{c["avg_searches"]:>9.2f} {c["avg_execution_time"]:>8.1f}')

print()
pos = m4c['POSITIVE_GRAPH']
pos_neg = m4c['POSITIVE_NEGATIVE_GRAPH']
print('=== Positive → Positive+Negative Delta ===')
print(f'  Tool calls:      {pos["avg_tool_calls"]:.2f} → {pos_neg["avg_tool_calls"]:.2f}  '
      f'({((pos["avg_tool_calls"]-pos_neg["avg_tool_calls"])/pos["avg_tool_calls"])*100:.1f}% reduction)')
print(f'  Searches:        {pos["avg_searches"]:.2f} → {pos_neg["avg_searches"]:.2f}  '
      f'({((pos["avg_searches"]-pos_neg["avg_searches"])/max(pos["avg_searches"],0.001))*100:.0f}% reduction)')
print(f'  Execution time:  {pos["avg_execution_time"]:.1f}s → {pos_neg["avg_execution_time"]:.1f}s  '
      f'({((pos["avg_execution_time"]-pos_neg["avg_execution_time"])/pos["avg_execution_time"])*100:.1f}% faster)')
print(f'  False neg guidance rate: {pos_neg["false_negative_guidance_rate"]:.1%}  (no over-pruning incidents)')

print()
print('=== SAFETY ABLATIONS ===')
print(f'{"Ablation":<35} {"Success":>8} {"ToolCalls":>10}')
print('-' * 55)
print(f'{"No Attribution Filter"::<35} {m4a["ABLATION_NO_ATTR"]["task_success_rate"]*100:>7.0f}%  '
      f'{m4a["ABLATION_NO_ATTR"]["avg_tool_calls"]:>9.1f}')
print(f'{"No Recency Decay"::<35} {m4a["ABLATION_NO_RECENCY"]["task_success_rate"]*100:>7.0f}%  '
      f'{m4a["ABLATION_NO_RECENCY"]["avg_tool_calls"]:>9.1f}')
print()
print('⚠ Ablation 1 (no attribution filter): 0% success — '
      'penalizing all visited edges poisons valid navigation paths.')

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('M4: Failure-Aware Graph — Results & Safety Ablation', fontsize=13, fontweight='bold')

    # Main conditions
    labels = ['A\nStateless', 'B\nStatic\nGraph', 'C\nPos\nGraph', 'D\nPos+Neg\nGraph ★', 'E\nPos+Neg\nNo Text']
    keys   = ['STATELESS','STATIC_GRAPH','POSITIVE_GRAPH','POSITIVE_NEGATIVE_GRAPH','POSITIVE_NEGATIVE_NO_TEXT']
    colors = ['#EA4335', '#4285F4', '#FBBC04', '#34A853', '#00BCD4']
    times  = [m4c[k]['avg_execution_time'] for k in keys]
    success= [m4c[k]['task_success_rate']*100 for k in keys]

    x = range(len(labels))
    bars = axes[0].bar(labels, times, color=colors, edgecolor='white')
    axes[0].set_title('Avg Execution Time (s) — M4 Conditions')
    axes[0].set_ylabel('Seconds')
    for bar, val, s in zip(bars, times, success):
        col = '#34A853' if s == 100 else '#EA4335'
        axes[0].text(bar.get_x()+bar.get_width()/2, val+1, f'{val:.0f}s\n({s:.0f}%)',
                     ha='center', fontsize=8, color=col, fontweight='bold')

    # Ablation chart
    abl_labels = ['Full M4\nSystem (D)', 'Ablation 1\nNo Attribution\nFilter', 'Ablation 2\nNo Recency\nDecay']
    abl_success= [100, 0, 0]
    abl_colors = ['#34A853', '#EA4335', '#EA4335']
    bars2 = axes[1].bar(abl_labels, abl_success, color=abl_colors, edgecolor='white', width=0.5)
    axes[1].set_title('Ablation Study: Task Success Rate')
    axes[1].set_ylabel('Success Rate (%)')
    axes[1].set_ylim(0, 120)
    for bar, val in zip(bars2, abl_success):
        axes[1].text(bar.get_x()+bar.get_width()/2, val+2, f'{val}%',
                     ha='center', fontsize=11, fontweight='bold')
    axes[1].axhline(100, color='gray', linestyle='--', alpha=0.4)
    axes[1].text(1, 60, 'Catastrophic\nfailure:\nover-pruning', ha='center',
                 color='#EA4335', fontsize=9, style='italic')

    plt.tight_layout()
    plt.savefig('m4_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: m4_results.png')

---
## 5. System Overview: Milestone Progression & Test Suite

Every milestone is validated by an independent unit test suite. All 121 tests pass with no regressions across milestones.

In [ ]:
milestones = [
    ('Bootstrap / M0', 'Environment setup, deterministic dev adapter', 9,  'test_core, test_agent, test_tools'),
    ('M1: Code Graph', 'AST parser, graph index, deterministic retrieval', 18, 'test_codegraph, test_retrieval, test_gemma_adapter'),
    ('M2: Experience', 'Episodic store, retriever, ranker, staleness', 9,  'test_experience'),
    ('M3: Exp Graph',  'Experience overlay, edge scorer, cold-start', 10, 'test_experience_graph'),
    ('M4: Neg Graph',  'Failure taxonomy, attribution, negative overlay', 10, 'test_negative_experience'),
    ('M5-A: Resources','ResourceRecord, provenance, permissions, SQLite', 24, 'test_resources'),
    ('M5-B: Ingestion','Adapters, validator, 7-stage pipeline', 15, 'test_adapters_and_validation'),
    ('M5-C: Res Graph','Resource graph, ranking, retriever', 15, 'test_resource_graph'),
    ('M5-D: Context',  '4-stream context builder, truth boundaries', 11, 'test_resource_context_and_reasoning'),
]

print('=== Test Suite: All Milestones ===')
print(f'{"Milestone":<18} {"Tests":>6}  {"Test Files"}')
print('-' * 80)
total = 0
for name, desc, count, files in milestones:
    print(f'{name:<18} {count:>6}  {files}')
    total += count
print('-' * 80)
print(f'{"TOTAL":<18} {total:>6}  all passing ✓')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 4))
    names  = [m[0] for m in milestones]
    counts = [m[2] for m in milestones]
    cum    = []
    running = 0
    for c in counts:
        running += c
        cum.append(running)

    palette = ['#4285F4','#4285F4','#FBBC04','#9334E6','#EA4335',
               '#34A853','#34A853','#34A853','#34A853']
    bars = ax.bar(names, counts, color=palette, edgecolor='white')
    ax2 = ax.twinx()
    ax2.plot(names, cum, 'k--o', markersize=5, label='Cumulative tests')
    ax2.set_ylabel('Cumulative tests')
    ax2.set_ylim(0, 140)

    for bar, val in zip(bars, counts):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.2, str(val),
                ha='center', fontsize=8, fontweight='bold')

    ax.set_xlabel('Milestone')
    ax.set_ylabel('New Tests Added')
    ax.set_title('LIFECORE-SWE: Test Suite Growth Across Milestones (121 total, all passing)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('test_suite.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: test_suite.png')

---
## 6. Key Takeaways

| Finding | Evidence |
|---|---|
| Graph-guided pre-retrieval is essential at scale | 0% → 100% success; stateless agent exhausts budget without finding target file |
| Structural memory replaces narrative text injection | Condition E: 100% success, 0 searches, fastest runtime — no text injected |
| Negative experience must be causally attributed | Ablation 1 (no filter): 0% success — indiscriminate penalization is catastrophic |
| Task-conditioned modulation prevents cross-module pollution | T4 rate-limiter edge utilization: 0.0% on unrelated transformer task |
| Architecture is model-agnostic | Graph overlays operate on AST topology + test outcomes, not model internals |

**The central scientific contribution:** Verified agent experience is more efficiently encoded as **structural weights on repository graph edges** than as narrative text in LLM prompts. The repository's AST graph acts as an externalized, compressed procedural memory.

In [ ]:
print('LIFECORE-SWE Notebook complete.')
print('All charts saved as PNG files in the working directory.')
print()
print('For full reproduction with live Gemma 4 runs:')
print('  export GOOGLE_API_KEY="your-key"')
print('  PYTHONPATH=. pytest -v tests/          # 121 tests, ~5s')
print('  PYTHONPATH=. python scripts/run_m3_sequence_experiment.py --condition EXP_GRAPH_NO_TEXT')
print('  PYTHONPATH=. python scripts/run_m4_failure_benchmark.py')